# 04 · Model the first Silver order slice

**Business question** · Can the parent and acquired dealer describe an order and its lines in the same way, with quantities and order amounts traceable to Bronze?

Run this notebook from the `silver` folder in VS Code using the same kernel and Bronze MinIO connection as notebooks 01–03. It reads existing snapshots and builds DataFrames in memory. Start with parent order `P-S1` and dealer order `D-S1`. Once they reconcile, use the contract below to design the Silver tables.

## 1 · Declare grain, keys, and relationships

| Entity | Grain | Keys and relationships |
| --- | --- | --- |
| Legal entity | One company that books transactions | `PARENT` and `DEALER` are local model identifiers. The configured ERPNext Company and the dedicated dealer database identify their respective legal scopes. |
| Source customer | One customer record per source and legal scope | Source customer ID. Many orders may reference the same customer. |
| Source product | One item record per source and legal scope | Source item code or dealer product ID. Many lines may reference the same product. |
| Warehouse | One physical location per legal entity | Source warehouse ID. An order line may reference one planned warehouse where captured. |
| Sales order | One order per source and legal entity | `(source_system, legal_entity_id, source_order_id)`. The order references the company and customer. |
| Sales order line | One distinct item line on an order | Order key plus `source_line_id`. Each line references its order and product. |

Keep header fields such as customer, order date, and currency at order grain. Keep product, quantity, unit, source unit price, and line amount at line grain. Maintain master names in their master tables. This preserves the order-to-line relationship without repeating header or master attributes in the line table.

## 2 · Map source fields into the canonical structure

| Field | Parent ERPNext | Dealer PostgreSQL |
| --- | --- | --- |
| Order identity and example lookup | `Sales Order.name` is the key. `custom_otc_seed_id` finds `P-S1`. | `sales_orders.id` is the key and identifies `D-S1`. |
| Customer and order date | `customer` and `transaction_date` | `customer_id` and `order_date` |
| Currency and status | `currency`, `status`, and `docstatus` | INR from the demo configuration. Status is absent from the header. |
| Line link and identity | Nested `items`, keyed by item-row `name` | `order_lines.order_id` links to the header. `id` identifies the line. |
| Product, quantity, and unit | `item_code`, `qty`, and transaction `uom` | `product_id` and `quantity`. The fixture uses `products.uom` as the order unit. |
| Line amount and unit price | `amount` and `rate` in transaction currency | `amount_minor` in paise. The unit price is absent. |
| Planned warehouse | Line `warehouse` | Absent on the order line. Fulfillment has its own warehouse relationship. |

The comparison uses recorded order line amounts. Revenue, shipment quantity, and invoice balances belong to later slices. Preserve the original unit and recorded amounts before applying any governed conversions.

In [1]:
from decimal import Decimal
from hashlib import sha256
import json
from urllib.parse import quote

from IPython.display import display
import pandas as pd

from bronze_review import decode_file, open_bronze

# Paste literal Airflow run IDs to reproduce a review. None selects the latest
# published snapshot independently for each source.
ERP_RUN_ID = None
DEALER_RUN_ID = None
# Pin the Bronze manifest key here if there are multiple seed-v1 deliveries.
MDM_MANIFEST_KEY = None
DEALER_CURRENCY = "INR"

bronze = open_bronze()


def verified_rows(key, sha, expected_rows=None):
    body = bronze.read(key)
    if sha256(body).hexdigest() != sha:
        raise ValueError(f"Checksum mismatch for {key}")
    rows, _ = decode_file(key, body)
    if rows is None or (expected_rows is not None and len(rows) != expected_rows):
        raise ValueError(f"Unexpected payload or row count for {key}")
    return rows


def load_snapshot(source, run_id, collections):
    prefix = f"batch/{source}/"
    if run_id is not None:
        manifest_key = f"{prefix}{quote(run_id, safe='')}/manifest.json"
        candidates = [manifest_key] if manifest_key in bronze.objects else []
    else:
        candidates = [
            key for key in bronze.objects
            if key.startswith(prefix) and key.endswith("/manifest.json")
        ]
    if not candidates:
        raise ValueError(f"No Bronze manifest for {source}, run={run_id!r}")
    manifests = [(key, json.loads(bronze.read(key))) for key in candidates]
    key, manifest = max(manifests, key=lambda pair: (pair[1]["published_at"], pair[0]))
    if manifest["source"] != source or (run_id is not None and manifest["run_id"] != run_id):
        raise ValueError(f"Unexpected source or run ID in {key}")
    entries = {row["collection"]: row for row in manifest["files"]}
    missing = set(collections) - entries.keys()
    if missing:
        raise ValueError(f"Missing collections in {key}: {sorted(missing)}")

    def read_collection(entry):
        return verified_rows(entry["key"], entry["sha256"], entry["rows"])

    records = {name: read_collection(entries[name]) for name in collections}
    trace = {"source": source, "run_id": manifest["run_id"], "manifest": key}
    return records, entries, trace


erp, erp_entries, erp_trace = load_snapshot(
    "erpnext", ERP_RUN_ID, ["Company", "Customer", "Item", "Warehouse", "Sales Order"]
)
dealer, dealer_entries, dealer_trace = load_snapshot(
    "dealer-sql", DEALER_RUN_ID, ["customers", "products", "warehouses", "sales_orders", "order_lines"]
)
display(pd.DataFrame([erp_trace, dealer_trace]))

,source,run_id,manifest
0,erpnext,manual__2026-11-22T12:12:42.982500+00:00,batch/erpnext/manual__2026-11-22T12%3A12%3A42....
1,dealer-sql,manual__2026-11-22T12:14:05.741782+00:00,batch/dealer-sql/manual__2026-11-22T12%3A14%3A...


In [2]:
def unique_index(records, field):
    index = {}
    for record in records:
        value = record[field]
        if value in (None, "") or value in index:
            raise ValueError(f"Blank or duplicate key in {field}: {value!r}")
        index[value] = record
    return index


def exactly_one(records, label):
    if len(records) != 1:
        raise ValueError(f"Expected one {label}, found {len(records)}")
    return records[0]


erp_company = exactly_one(erp["Company"], "ERPNext Company")
erp_customer = unique_index(erp["Customer"], "name")
erp_product = unique_index(erp["Item"], "item_code")
erp_warehouse = unique_index(
    [row for row in erp["Warehouse"] if not row["is_group"]], "name"
)
dealer_customer = unique_index(dealer["customers"], "id")
dealer_product = unique_index(dealer["products"], "id")
dealer_warehouse = unique_index(dealer["warehouses"], "id")

legal_entities = pd.DataFrame([
    {"legal_entity_id": "PARENT", "source_system": "erpnext", "source_scope": erp_company["name"]},
    {"legal_entity_id": "DEALER", "source_system": "dealer-sql", "source_scope": "dealer_legacy"},
])


def master_frame(parent, acquired, parent_name, acquired_name):
    return pd.DataFrame([
        {"source_system": source, "legal_entity_id": entity, "source_id": key, "name": row[label]}
        for source, entity, index, label in [
            ("erpnext", "PARENT", parent, parent_name),
            ("dealer-sql", "DEALER", acquired, acquired_name),
        ]
        for key, row in index.items()
    ])


customers = master_frame(erp_customer, dealer_customer, "customer_name", "name")
products = master_frame(erp_product, dealer_product, "item_name", "name")
warehouses = master_frame(erp_warehouse, dealer_warehouse, "warehouse_name", "name")
display(legal_entities, customers, products, warehouses)

,legal_entity_id,source_system,source_scope
0,PARENT,erpnext,OTC Parent Demo
1,DEALER,dealer-sql,dealer_legacy


,source_system,legal_entity_id,source_id,name
0,erpnext,PARENT,OTC P-C1 Metro Industrial Services,OTC P-C1 Metro Industrial Services
1,erpnext,PARENT,OTC P-C2 Pioneer Engineering,OTC P-C2 Pioneer Engineering
2,erpnext,PARENT,OTC P-C3 Apex Industrial Supply,OTC P-C3 Apex Industrial Supply
3,erpnext,PARENT,OTC P-C4 Summit Process Systems,OTC P-C4 Summit Process Systems
4,erpnext,PARENT,OTC P-C5 Harbor Utilities,OTC P-C5 Harbor Utilities
5,dealer-sql,DEALER,D-C1,Metro Ind. Services
6,dealer-sql,DEALER,D-C2,Pioneer Engineering
7,dealer-sql,DEALER,D-C3,Apex Industrial
8,dealer-sql,DEALER,D-C4,Summit Process
9,dealer-sql,DEALER,D-C5,Valley Fabrication


,source_system,legal_entity_id,source_id,name
0,erpnext,PARENT,P-FILT,OTC Filter
1,erpnext,PARENT,P-GAUGE,OTC Pressure gauge
2,erpnext,PARENT,P-PUMP,OTC Replacement pump
3,erpnext,PARENT,P-SEAL,OTC Seal
4,erpnext,PARENT,P-VALVE,OTC Valve
5,dealer-sql,DEALER,D-FLT,Filter
6,dealer-sql,DEALER,D-HOSE,Hydraulic hose
7,dealer-sql,DEALER,D-PUMP,Replacement pump
8,dealer-sql,DEALER,D-SEAL,Seal
9,dealer-sql,DEALER,D-VALVE,Valve


,source_system,legal_entity_id,source_id,name
0,erpnext,PARENT,Finished Goods - OPD,Finished Goods
1,erpnext,PARENT,Goods In Transit - OPD,Goods In Transit
2,erpnext,PARENT,P-WH1 - OPD,P-WH1
3,erpnext,PARENT,P-WH2 - OPD,P-WH2
4,erpnext,PARENT,Stores - OPD,Stores
5,erpnext,PARENT,Work In Progress - OPD,Work In Progress
6,dealer-sql,DEALER,D-WH1,Dealer distribution


In [3]:
parent = exactly_one(
    [row for row in erp["Sales Order"] if row.get("custom_otc_seed_id") == "P-S1"],
    "ERPNext Sales Order with seed ID P-S1",
)
acquired = unique_index(dealer["sales_orders"], "id")["D-S1"]
if parent["company"] != erp_company["name"]:
    raise ValueError("P-S1 company differs from the selected ERPNext company")


def amount(value):
    result = Decimal(str(value))
    if not result.is_finite():
        raise ValueError(f"Invalid numeric value: {value!r}")
    return result


order_key = ["source_system", "legal_entity_id", "source_order_id"]
parent_key = dict(zip(order_key, ("erpnext", "PARENT", parent["name"])))
dealer_key = dict(zip(order_key, ("dealer-sql", "DEALER", acquired["id"])))

sales_order = pd.DataFrame([
    {**parent_key, "order_reference": "P-S1", "source_customer_id": parent["customer"],
     "order_date": parent["transaction_date"], "currency": parent["currency"],
     "source_status": parent.get("status"), "source_docstatus": parent["docstatus"],
     "bronze_key": erp_entries["Sales Order"]["key"]},
    {**dealer_key, "order_reference": "D-S1", "source_customer_id": acquired["customer_id"],
     "order_date": acquired["order_date"], "currency": DEALER_CURRENCY,
     "source_status": None, "source_docstatus": None,
     "bronze_key": dealer_entries["sales_orders"]["key"]},
])

parent_lines = list(unique_index(parent["items"], "name").values())
dealer_lines = list(unique_index(
    [row for row in dealer["order_lines"] if row["order_id"] == acquired["id"]], "id"
).values())

sales_order_line = pd.DataFrame([
    {**parent_key, "source_line_id": row["name"], "source_product_id": row["item_code"],
     "quantity": amount(row["qty"]), "source_uom": row["uom"],
     "source_unit_price": amount(row["rate"]), "line_amount": amount(row["amount"]),
     "source_warehouse_id": row.get("warehouse") or None,
     "bronze_key": erp_entries["Sales Order"]["key"]}
    for row in parent_lines
] + [
    {**dealer_key, "source_line_id": row["id"], "source_product_id": row["product_id"],
     "quantity": amount(row["quantity"]), "source_uom": dealer_product[row["product_id"]]["uom"],
     "source_unit_price": None, "line_amount": amount(row["amount_minor"]).scaleb(-2),
     "source_warehouse_id": None, "bronze_key": dealer_entries["order_lines"]["key"]}
    for row in dealer_lines
])
display(sales_order, sales_order_line)

,source_system,legal_entity_id,source_order_id,order_reference,source_customer_id,order_date,currency,source_status,source_docstatus,bronze_key
0,erpnext,PARENT,SAL-ORD-2026-00001,P-S1,OTC P-C1 Metro Industrial Services,2026-11-02,INR,Completed,1.0,batch/erpnext/manual__2026-11-22T12%3A12%3A42....
1,dealer-sql,DEALER,D-S1,D-S1,D-C1,2026-11-05,INR,None,NaN,batch/dealer-sql/manual__2026-11-22T12%3A14%3A...


,source_system,legal_entity_id,source_order_id,source_line_id,source_product_id,quantity,source_uom,source_unit_price,line_amount,source_warehouse_id,bronze_key
0,erpnext,PARENT,SAL-ORD-2026-00001,vqgdj6mnv1,P-PUMP,10.0,Nos,10000.0,100000.0,P-WH1 - OPD,batch/erpnext/manual__2026-11-22T12%3A12%3A42....
1,dealer-sql,DEALER,D-S1,D-L1,D-PUMP,1,Each,None,40000.00,None,batch/dealer-sql/manual__2026-11-22T12%3A14%3A...


## 3 · Reconcile before expanding the model

Expect ten pumps and ₹100,000 for P-S1, and one pump and ₹40,000 for D-S1. Validate counts, amounts, and references at each order's grain.

The MDM example uses the published demo mapping fixture. It resolves approved product matches for inspection and retains review cases with a blank shared ID. Customer conformance needs a verified bridge from the ERPNext customer ID to the fixture alias. Warehouse identity remains scoped to the owning company. Future effective-dated crosswalk and unit-conversion tables can be designed independently of the order header and line tables.

In [4]:
if sales_order.duplicated(order_key).any():
    raise ValueError("Duplicate Silver order key")
if sales_order_line.duplicated(order_key + ["source_line_id"]).any():
    raise ValueError("Duplicate Silver order line key")

reconciliation = []
for key, customer_index, product_index, warehouse_index, raw_lines, qty_field, raw_amounts, control in [
    (parent_key, erp_customer, erp_product, erp_warehouse, parent_lines, "qty",
     [amount(row["amount"]) for row in parent_lines], (10, 100000)),
    (dealer_key, dealer_customer, dealer_product, dealer_warehouse, dealer_lines, "quantity",
     [amount(row["amount_minor"]).scaleb(-2) for row in dealer_lines], (1, 40000)),
]:
    order = sales_order.loc[sales_order[order_key].eq(key).all(axis=1)].iloc[0]
    lines = sales_order_line.loc[sales_order_line[order_key].eq(key).all(axis=1)]
    quantity = sum(lines["quantity"])
    total = sum(lines["line_amount"])
    source_quantity = sum(amount(row[qty_field]) for row in raw_lines)
    source_total = sum(raw_amounts)
    references_valid = (
        order["source_customer_id"] in customer_index
        and lines["source_product_id"].isin(product_index).all()
        and lines["source_warehouse_id"].dropna().isin(warehouse_index).all()
    )
    passed = (
        references_valid and len(lines) == len(raw_lines) == 1
        and quantity == source_quantity == control[0]
        and total == source_total == control[1] and order["currency"] == "INR"
    )
    reconciliation.append({
        "order_reference": order["order_reference"], "bronze_lines": len(raw_lines),
        "silver_lines": len(lines), "bronze_quantity": source_quantity,
        "silver_quantity": quantity, "bronze_amount": source_total, "silver_amount": total,
        "currency": order["currency"], "references_valid": references_valid,
        "result": "PASS" if passed else "FAIL",
    })

reconciliation = pd.DataFrame(reconciliation)
display(reconciliation)
if not reconciliation["result"].eq("PASS").all():
    raise ValueError("Reconciliation failed. Review the controls and pinned manifest keys above.")

,order_reference,bronze_lines,silver_lines,bronze_quantity,silver_quantity,bronze_amount,silver_amount,currency,references_valid,result
0,P-S1,1,1,10.0,10.0,100000.0,100000.0,INR,True,PASS
1,D-S1,1,1,1,1,40000.00,40000.00,INR,True,PASS


In [5]:
candidate_source_key = "seed-v1/mdm/candidates.json"
manifest_keys = [
    key for key in bronze.objects
    if key.startswith("files/otc-inbound/seed-v1/") and key.endswith("/manifest.json")
]
if MDM_MANIFEST_KEY is not None:
    if MDM_MANIFEST_KEY not in manifest_keys:
        raise ValueError("MDM_MANIFEST_KEY must identify a published seed-v1 manifest")
    manifest_keys = [MDM_MANIFEST_KEY]

if not manifest_keys:
    print("No seed-v1 MDM manifest found. The two canonical orders remain available above.")
else:
    mdm_key = exactly_one(manifest_keys, "seed-v1 MDM manifest (set MDM_MANIFEST_KEY if multiple)")
    manifest = json.loads(bronze.read(mdm_key))
    entry = exactly_one(
        [row for row in manifest["files"] if row["key"] == candidate_source_key],
        "product and customer candidate payload",
    )
    key = f"{mdm_key.rsplit('/', 1)[0]}/objects/{candidate_source_key}"
    candidates = verified_rows(key, entry["sha256"])
    crosswalk_rows = []
    for candidate in candidates:
        if candidate["domain"] != "product":
            continue
        for side, source, entity, own_group_field in [
            ("parent", "erpnext", "PARENT", "expected_group_id"),
            ("dealer", "dealer-sql", "DEALER", "expected_dealer_group_id"),
        ]:
            state = candidate["status"]
            if state not in {"approved", "pending", "separate"}:
                raise ValueError(f"Unexpected product mapping state: {state!r}")
            crosswalk_rows.append({
                "source_system": source, "legal_entity_id": entity,
                "source_product_id": candidate[side], "mapping_status": state,
                "group_product_id": (
                    candidate["expected_group_id"] if state == "approved"
                    else candidate[own_group_field] if state == "separate" else None
                ),
            })
    product_keys = ["source_system", "legal_entity_id", "source_product_id"]
    crosswalk = pd.DataFrame(crosswalk_rows, columns=[
        *product_keys, "mapping_status", "group_product_id",
    ])
    product_mapping = products.rename(columns={"source_id": "source_product_id"}).merge(
        crosswalk, on=product_keys, how="left", validate="one_to_one"
    )
    product_mapping["mapping_status"] = product_mapping["mapping_status"].fillna("unmapped")
    order_line_mapping = sales_order_line.merge(
        product_mapping[product_keys + ["mapping_status", "group_product_id"]],
        on=product_keys, how="left", validate="many_to_one",
    )
    display(product_mapping, order_line_mapping)
    pending = product_mapping["mapping_status"].eq("pending")
    if not pending.any() or product_mapping.loc[pending, "group_product_id"].notna().any():
        raise ValueError("Expected pending product candidates with no shared ID")
    if (not order_line_mapping["mapping_status"].eq("approved").all()
            or order_line_mapping["group_product_id"].isna().any()
            or order_line_mapping["group_product_id"].nunique() != 1):
        raise ValueError("Expected both pump lines to share one approved product identity")
    print("MDM PASS · Both pump lines share an approved ID and review cases retain their source IDs.")

,source_system,legal_entity_id,source_product_id,name,mapping_status,group_product_id
0,erpnext,PARENT,P-FILT,OTC Filter,pending,None
1,erpnext,PARENT,P-GAUGE,OTC Pressure gauge,separate,G-PROD-005
2,erpnext,PARENT,P-PUMP,OTC Replacement pump,approved,G-PROD-001
3,erpnext,PARENT,P-SEAL,OTC Seal,pending,None
4,erpnext,PARENT,P-VALVE,OTC Valve,approved,G-PROD-002
5,dealer-sql,DEALER,D-FLT,Filter,pending,None
6,dealer-sql,DEALER,D-HOSE,Hydraulic hose,separate,G-PROD-006
7,dealer-sql,DEALER,D-PUMP,Replacement pump,approved,G-PROD-001
8,dealer-sql,DEALER,D-SEAL,Seal,pending,None
9,dealer-sql,DEALER,D-VALVE,Valve,approved,G-PROD-002


,source_system,legal_entity_id,source_order_id,source_line_id,source_product_id,quantity,source_uom,source_unit_price,line_amount,source_warehouse_id,bronze_key,mapping_status,group_product_id
0,erpnext,PARENT,SAL-ORD-2026-00001,vqgdj6mnv1,P-PUMP,10.0,Nos,10000.0,100000.0,P-WH1 - OPD,batch/erpnext/manual__2026-11-22T12%3A12%3A42....,approved,G-PROD-001
1,dealer-sql,DEALER,D-S1,D-L1,D-PUMP,1,Each,None,40000.00,None,batch/dealer-sql/manual__2026-11-22T12%3A14%3A...,approved,G-PROD-001


MDM PASS · Both pump lines share an approved ID and review cases retain their source IDs.


## Review before designing fulfillment

Confirm the two source run IDs, the scoped primary keys, and the reconciliation totals. The notebook reads a snapshot per source and builds a current-state example. For implementation, define when an order update creates a historical version and keep business-effective time distinct from Bronze observation time. Use the scoped order-line key when modeling partial fulfillment. Start with P-S1, whose two delivery notes let you verify the one-to-many relationship.